# SEC EDGAR Raw Data Test

**Phase 1 — No cloud, no Snowflake. Pure HTTP → DataFrame.**

Tests all 3 public SEC EDGAR endpoints:
1. `company_tickers_exchange.json` — full ticker/exchange snapshot
2. `submissions/CIK{cik}.json` — company metadata (SIC, entity type, filings list)
3. `api/xbrl/companyfacts/CIK{cik}.json` — XBRL financial facts

**Mandatory SEC compliance:**
- `User-Agent` header on every request (SEC Developer Resources requirement)
- Rate limited to ≤8 req/s (SEC hard limit is 10 req/s per IP)

In [1]:
# ── 0. Dependencies (uv) ─────────────────────────────────────────────────────
# From the repo root in PowerShell (run once):
#
#   uv sync --group dev
#   uv run jupyter notebook notebooks\sec_edgar_raw_test.ipynb
#
# uv reads pyproject.toml + .python-version and creates .venv automatically.
# No Activate.ps1 needed -- `uv run` handles the venv.
#
# Install uv on Windows (once, system-wide):
#   powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"

In [2]:
import os, json, time, threading, sys
from datetime import date, datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from requests.adapters import HTTPAdapter, Retry
import pandas as pd

# ── SEC User-Agent (required by SEC Developer Resources) ─────────────────────
os.environ["SEC_USER_AGENT"] = "n/a prototyping paul.ananth@yahoo.com"

USER_AGENT = os.environ["SEC_USER_AGENT"]
print(f"User-Agent : {USER_AGENT}")
print(f"Date       : {date.today()}")

User-Agent : n/a prototyping paul.ananth@yahoo.com
Date       : 2026-04-07


In [3]:
# ── Rate limiter (token-bucket, 8 req/s) ─────────────────────────────────────
class RateLimiter:
    """Thread-safe token-bucket. Caps at 8 req/s (20% below SEC's 10 req/s limit)."""
    def __init__(self, rps=8.0):
        self._rps, self._tokens, self._last = rps, rps, time.monotonic()
        self._lock = threading.Lock()
    def acquire(self):
        with self._lock:
            now = time.monotonic()
            self._tokens = min(self._rps, self._tokens + (now - self._last) * self._rps)
            self._last = now
            if self._tokens < 1.0:
                time.sleep((1.0 - self._tokens) / self._rps)
                self._tokens = 0.0
            else:
                self._tokens -= 1.0

# ── HTTP session (User-Agent injected on every request) ──────────────────────
_retry = Retry(total=3, backoff_factor=1.0, status_forcelist=[429, 500, 502, 503])
_session = requests.Session()
_session.headers["User-Agent"] = USER_AGENT
_session.mount("https://", HTTPAdapter(max_retries=_retry))

_limiter = RateLimiter(rps=8.0)

def fetch(url):
    """Rate-limited GET. Returns parsed JSON or None on 404."""
    _limiter.acquire()
    r = _session.get(url, timeout=20)
    if r.status_code == 404:
        return None
    r.raise_for_status()
    return r.json()

print("HTTP client ready  (User-Agent set, rate limiter 8 req/s)")

HTTP client ready  (User-Agent set, rate limiter 8 req/s)


---
## 1. Ticker Exchange Snapshot
Single daily file — all tickers listed on NYSE and Nasdaq (~10,000 rows).

In [4]:
raw = fetch("https://www.sec.gov/files/company_tickers_exchange.json")

tickers_df = pd.DataFrame(raw["data"], columns=raw["fields"])
tickers_df["cik"] = tickers_df["cik"].astype(str).str.zfill(10)

print(f"Total rows : {len(tickers_df):,}")
print(f"Exchanges  : {tickers_df['exchange'].value_counts().to_dict()}")
tickers_df.head(10)

Total rows : 10,433
Exchanges  : {'Nasdaq': 4248, 'NYSE': 3273, 'OTC': 2596, 'CBOE': 28}


,cik,name,ticker,exchange
0,0001045810,NVIDIA CORP,NVDA,Nasdaq
1,0000320193,Apple Inc.,AAPL,Nasdaq
2,0001652044,Alphabet Inc.,GOOGL,Nasdaq
3,0000789019,MICROSOFT CORP,MSFT,Nasdaq
4,0001018724,AMAZON COM INC,AMZN,Nasdaq
5,0001730168,Broadcom Inc.,AVGO,Nasdaq
6,0001318605,"Tesla, Inc.",TSLA,Nasdaq
7,0001326801,"Meta Platforms, Inc.",META,Nasdaq
8,0001067983,BERKSHIRE HATHAWAY INC,BRK-B,NYSE
9,0000104169,Walmart Inc.,WMT,Nasdaq


In [5]:
# Filter to NYSE + Nasdaq only
listed = tickers_df[tickers_df["exchange"].isin(["NYSE", "Nasdaq"])].copy()
print(f"NYSE + Nasdaq : {len(listed):,} tickers")
listed.sample(10)

NYSE + Nasdaq : 7,521 tickers


,cik,name,ticker,exchange
1345,0001164863,Enpro Inc.,NPO,NYSE
1882,0001877787,Ermenegildo Zegna N.V.,ZGN,NYSE
1267,0001650729,"SiteOne Landscape Supply, Inc.",SITE,NYSE
1428,0000813762,ICAHN ENTERPRISES L.P.,IEP,Nasdaq
9139,0001601712,Synchrony Financial,SYF-PB,NYSE
4165,0002041047,Dune Acquisition Corp II,IPOD,Nasdaq
4477,0001645873,"MODIV INDUSTRIAL, INC.",MDV,NYSE
2718,0001827090,"Certara, Inc.",CERT,Nasdaq
2713,0000031235,EASTMAN KODAK CO,KODK,NYSE
6676,0001828673,HCW Biologics Inc.,HCWB,Nasdaq


---
## 2. Submissions — Full Company Record (ALL fields)

One JSON file per CIK. Contains every field the SEC stores about the registrant:
company metadata, addresses, former names, fiscal year end, insider transaction flags —
plus a full filings index (`filings.recent`) stored as columnar parallel arrays.

**No fields are dropped.** `Submission` Pydantic model maps every top-level key.
`filings.recent` is exploded from columnar format → individual `Filing` rows.

In [6]:
from pathlib import Path

repo_root = Path.cwd().resolve()
if not (repo_root / "scripts").exists():
    repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scripts.ingest.models import Submission, explode_filings

# Pilot set — 10 well-known companies
PILOT = [
    ("AAPL",  "0000320193"),
    ("MSFT",  "0000789019"),
    ("AMZN",  "0001018724"),
    ("GOOG",  "0001652044"),
    ("META",  "0001326801"),
    ("TSLA",  "0001318605"),
    ("JPM",   "0000019617"),
    ("JNJ",   "0000200406"),
    ("XOM",   "0000034088"),
    ("BRK-B", "0001067983"),
]

submissions_meta = []   # one row per company (all top-level fields)
all_filings      = []   # one row per filing (filings.recent exploded)

for ticker, cik10 in PILOT:
    raw = fetch(f"https://data.sec.gov/submissions/CIK{cik10}.json")
    if raw is None:
        print(f"  {ticker:<8} 404")
        continue

    # Validate the full response — raises ValidationError if SEC changes the schema
    sub = Submission.model_validate(raw)

    # ── Company-level metadata (all top-level fields) ────────────────────────
    meta = sub.model_dump(exclude={"filing_files"})
    meta["ticker"]         = ticker
    meta["filing_files_count"] = len(sub.filing_files)
    # Flatten addresses to individual columns
    for side in ("mailing", "business"):
        addr = (sub.addresses or {}).get(side)
        if addr:
            for k, v in addr.model_dump().items():
                meta[f"addr_{side}_{k}"] = v
    del meta["addresses"]   # replaced by flat columns above
    submissions_meta.append(meta)

    # ── Filings index (filings.recent exploded to rows) ──────────────────────
    recent = raw.get("filings", {}).get("recent", {})
    filing_rows = explode_filings(cik10, recent)
    for r in filing_rows:
        r["ticker"] = ticker
    all_filings.extend(filing_rows)

    print(f"  {ticker:<8} {sub.name[:45]:<45}  "
          f"fisc_yr_end={sub.fiscalYearEnd}  "
          f"filings={len(filing_rows)}")

submissions_df = pd.DataFrame(submissions_meta)
filings_df     = pd.DataFrame(all_filings)

print(f"\nsubmissions_df : {submissions_df.shape[0]} rows × {submissions_df.shape[1]} columns")
print(f"filings_df     : {filings_df.shape[0]} rows × {filings_df.shape[1]} columns")

  AAPL     Apple Inc.                                     fisc_yr_end=0926  filings=1000
  MSFT     MICROSOFT CORP                                 fisc_yr_end=1231  filings=1008
  AMZN     AMAZON COM INC                                 fisc_yr_end=1231  filings=1009
  GOOG     Alphabet Inc.                                  fisc_yr_end=1231  filings=1016
  META     Meta Platforms, Inc.                           fisc_yr_end=1231  filings=1002
  TSLA     Tesla, Inc.                                    fisc_yr_end=1231  filings=1001
  JPM      JPMORGAN CHASE & CO                            fisc_yr_end=1231  filings=23638
  JNJ      JOHNSON & JOHNSON                              fisc_yr_end=1228  filings=1004
  XOM      EXXON MOBIL CORP                               fisc_yr_end=1231  filings=1000
  BRK-B    BERKSHIRE HATHAWAY INC                         fisc_yr_end=1231  filings=1000

submissions_df : 10 rows × 42 columns
filings_df     : 32678 rows × 18 columns


In [7]:
# Company metadata — all fields
print("Columns:", submissions_df.columns.tolist())
submissions_df

Columns: ['cik', 'entityType', 'sic', 'sicDescription', 'ownerOrg', 'insiderTransactionForOwnerExists', 'insiderTransactionForIssuerExists', 'name', 'tickers', 'exchanges', 'ein', 'lei', 'description', 'website', 'investorWebsite', 'category', 'fiscalYearEnd', 'stateOfIncorporation', 'stateOfIncorporationDescription', 'phone', 'flags', 'formerNames', 'ticker', 'filing_files_count', 'addr_mailing_street1', 'addr_mailing_street2', 'addr_mailing_city', 'addr_mailing_stateOrCountry', 'addr_mailing_zipCode', 'addr_mailing_stateOrCountryDescription', 'addr_mailing_isForeignLocation', 'addr_mailing_foreignStateTerritory', 'addr_mailing_country', 'addr_business_street1', 'addr_business_street2', 'addr_business_city', 'addr_business_stateOrCountry', 'addr_business_zipCode', 'addr_business_stateOrCountryDescription', 'addr_business_isForeignLocation', 'addr_business_foreignStateTerritory', 'addr_business_country']


,cik,entityType,sic,sicDescription,ownerOrg,insiderTransactionForOwnerExists,insiderTransactionForIssuerExists,name,tickers,exchanges,...,addr_mailing_country,addr_business_street1,addr_business_street2,addr_business_city,addr_business_stateOrCountry,addr_business_zipCode,addr_business_stateOrCountryDescription,addr_business_isForeignLocation,addr_business_foreignStateTerritory,addr_business_country
0,0000320193,operating,3571,Electronic Computers,06 Technology,0,1,Apple Inc.,[AAPL],[Nasdaq],...,None,ONE APPLE PARK WAY,None,CUPERTINO,CA,95014,CA,None,None,None
1,0000789019,operating,7372,Services-Prepackaged Software,06 Technology,1,1,MICROSOFT CORP,[MSFT],[Nasdaq],...,None,ONE MICROSOFT WAY,None,REDMOND,WA,98052-6399,WA,None,None,None
2,0001018724,operating,5961,Retail-Catalog & Mail-Order Houses,07 Trade & Services,1,1,AMAZON COM INC,[AMZN],[Nasdaq],...,None,410 TERRY AVENUE NORTH,None,SEATTLE,WA,98109,WA,None,None,None
3,0001652044,operating,7370,"Services-Computer Programming, Data Processing...",06 Technology,1,1,Alphabet Inc.,"[GOOGL, GOOG]","[Nasdaq, Nasdaq]",...,None,1600 AMPHITHEATRE PARKWAY,None,MOUNTAIN VIEW,CA,94043,CA,None,None,None
4,0001326801,operating,7370,"Services-Computer Programming, Data Processing...",06 Technology,0,1,"Meta Platforms, Inc.",[META],[Nasdaq],...,None,1 META WAY,None,MENLO PARK,CA,94025,CA,None,None,None
5,0001318605,operating,3711,Motor Vehicles & Passenger Car Bodies,04 Manufacturing,0,1,"Tesla, Inc.",[TSLA],[Nasdaq],...,None,1 TESLA ROAD,None,AUSTIN,TX,78725,TX,None,None,None
6,0000019617,operating,6021,National Commercial Banks,02 Finance,1,1,JPMORGAN CHASE & CO,"[JPM, JPM-PC, JPM-PD, AMJB, JPM-PJ, JPM-PK, JP...","[NYSE, NYSE, NYSE, NYSE, NYSE, NYSE, NYSE, NYS...",...,None,270 PARK AVENUE,None,NEW YORK,NY,10017,NY,None,None,None
7,0000200406,operating,2834,Pharmaceutical Preparations,03 Life Sciences,1,1,JOHNSON & JOHNSON,[JNJ],[NYSE],...,None,ONE JOHNSON & JOHNSON PLZ,None,NEW BRUNSWICK,NJ,08933,NJ,None,None,None
8,0000034088,operating,2911,Petroleum Refining,01 Energy & Transportation,1,1,EXXON MOBIL CORP,[XOM],[NYSE],...,None,22777 SPRINGWOODS VILLAGE PARKWAY,None,SPRING,TX,77389-1425,TX,None,None,None
9,0001067983,operating,6331,"Fire, Marine & Casualty Insurance",02 Finance,1,1,BERKSHIRE HATHAWAY INC,"[BRK-B, BRK-A]","[NYSE, NYSE]",...,None,3555 FARNAM STREET,None,OMAHA,NE,68131,NE,None,None,None


In [8]:
# Filings index — one row per filing (accessionNumber, form, isXBRL, size, primaryDocument)
print(f"Total filings across {len(PILOT)} companies: {len(filings_df):,}")
filings_df[["ticker", "cik", "accessionNumber", "filingDate", "form",
            "isXBRL", "isInlineXBRL", "size", "primaryDocument"]].head(20)

Total filings across 10 companies: 32,678


,ticker,cik,accessionNumber,filingDate,form,isXBRL,isInlineXBRL,size,primaryDocument
0,AAPL,0000320193,0001140361-26-013192,2026-04-03,4,0,0,17348,xslF345X06/form4.xml
1,AAPL,0000320193,0001140361-26-013191,2026-04-03,4,0,0,14156,xslF345X06/form4.xml
2,AAPL,0000320193,0001140361-26-013190,2026-04-03,4,0,0,25322,xslF345X06/form4.xml
3,AAPL,0000320193,0001969223-26-000420,2026-04-02,144,0,0,4439,xsl144X01/primary_doc.xml
4,AAPL,0000320193,0001959173-26-002757,2026-04-02,144,0,0,4652,xsl144X01/primary_doc.xml
5,AAPL,0000320193,0000102909-26-000630,2026-03-26,SCHEDULE 13G/A,0,0,7205,xslSCHEDULE_13G_X02/primary_doc.xml
6,AAPL,0000320193,0001780525-26-000005,2026-03-17,4,0,0,9251,xslF345X05/wk-form4_1773786674.xml
7,AAPL,0000320193,0001780525-26-000003,2026-03-06,3,0,0,490535,xslF345X02/wk-form3_1772839848.xml
8,AAPL,0000320193,0001059235-26-000004,2026-02-26,4,0,0,5753,xslF345X05/wk-form4_1772148856.xml
9,AAPL,0000320193,0001216519-26-000004,2026-02-26,4,0,0,5760,xslF345X05/wk-form4_1772148826.xml


---
## 3. XBRL Company Facts — ALL Concepts, ALL Namespaces (Raw)

The `companyfacts` endpoint contains every tagged financial value ever filed.
A large company (AAPL) has ~3,000+ XBRL concepts across three namespaces:
- **`us-gaap`** — US GAAP financial statement items
- **`dei`** — Document & Entity Information (shares outstanding, fiscal year end, etc.)
- **`ifrs-full`** — IFRS equivalents (non-US filers)

**No filtering.** Every namespace, every concept, every unit, every entry is kept.
The `frame` column (e.g. `CY2023`, `CY2023Q3I`) is preserved — it identifies
SEC-standardised calendar-period tags useful for cross-company comparison.

In [9]:
from scripts.ingest.models import CompanyFacts, explode_facts

all_facts = []

# Sequential fetch — companyfacts payloads are 5–15 MB each
for ticker, cik10 in PILOT:
    raw = fetch(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik10}.json")
    if raw is None:
        print(f"  {ticker:<8} 404")
        continue

    # Validate full structure via Pydantic
    parsed = CompanyFacts.model_validate(raw)

    # Explode ALL namespaces / concepts / entries — no filter
    rows = explode_facts(parsed)
    for r in rows:
        r["ticker"] = ticker
    all_facts.extend(rows)

    namespaces = list(parsed.facts.keys())
    concepts_per_ns = {ns: len(c) for ns, c in parsed.facts.items()}
    print(f"  {ticker:<8}  {len(rows):>6,} rows  |  namespaces: {namespaces}  |  concepts: {concepts_per_ns}")

facts_df = pd.DataFrame(all_facts)
facts_df["val"]   = pd.to_numeric(facts_df["val"], errors="coerce")
facts_df["end"]   = pd.to_datetime(facts_df["end"], errors="coerce")
facts_df["filed"] = pd.to_datetime(facts_df["filed"], errors="coerce")
# start is None for instant facts — keep as object column
print(f"\nTotal raw fact rows : {len(facts_df):,}")
print(f"Columns             : {facts_df.columns.tolist()}")

  AAPL      24,579 rows  |  namespaces: ['dei', 'us-gaap']  |  concepts: {'dei': 2, 'us-gaap': 503}
  MSFT      31,574 rows  |  namespaces: ['dei', 'us-gaap']  |  concepts: {'dei': 3, 'us-gaap': 543}
  AMZN      28,812 rows  |  namespaces: ['dei', 'us-gaap']  |  concepts: {'dei': 2, 'us-gaap': 538}
  GOOG      19,836 rows  |  namespaces: ['dei', 'us-gaap']  |  concepts: {'dei': 1, 'us-gaap': 521}
  META      17,446 rows  |  namespaces: ['dei', 'us-gaap', 'srt']  |  concepts: {'dei': 1, 'us-gaap': 455, 'srt': 1}
  TSLA      23,454 rows  |  namespaces: ['dei', 'us-gaap']  |  concepts: {'dei': 2, 'us-gaap': 638}
  JPM       48,746 rows  |  namespaces: ['dei', 'invest', 'srt', 'us-gaap', 'ffd', 'ecd']  |  concepts: {'dei': 2, 'invest': 1, 'srt': 2, 'us-gaap': 917, 'ffd': 1, 'ecd': 7}
  JNJ       26,863 rows  |  namespaces: ['dei', 'us-gaap']  |  concepts: {'dei': 2, 'us-gaap': 608}
  XOM       20,399 rows  |  namespaces: ['dei', 'us-gaap']  |  concepts: {'dei': 2, 'us-gaap': 438}
  BRK-B  

In [10]:
facts_df.head(10)

,cik,entity_name,namespace,concept,label,unit,end,start,val,accn,form,filed,frame,ticker
0,0000320193,Apple Inc.,dei,EntityCommonStockSharesOutstanding,"Entity Common Stock, Shares Outstanding",shares,2009-06-27,NaN,895816758.0,0001193125-09-153165,10-Q,2009-07-22,CY2009Q2I,AAPL
1,0000320193,Apple Inc.,dei,EntityCommonStockSharesOutstanding,"Entity Common Stock, Shares Outstanding",shares,2009-10-16,NaN,900678473.0,0001193125-09-214859,10-K,2009-10-27,NaN,AAPL
2,0000320193,Apple Inc.,dei,EntityCommonStockSharesOutstanding,"Entity Common Stock, Shares Outstanding",shares,2009-10-16,NaN,900678473.0,0001193125-10-012091,10-K/A,2010-01-25,CY2009Q3I,AAPL
3,0000320193,Apple Inc.,dei,EntityCommonStockSharesOutstanding,"Entity Common Stock, Shares Outstanding",shares,2010-01-15,NaN,906794589.0,0001193125-10-012085,10-Q,2010-01-25,CY2009Q4I,AAPL
4,0000320193,Apple Inc.,dei,EntityCommonStockSharesOutstanding,"Entity Common Stock, Shares Outstanding",shares,2010-04-09,NaN,909938383.0,0001193125-10-088957,10-Q,2010-04-21,CY2010Q1I,AAPL
5,0000320193,Apple Inc.,dei,EntityCommonStockSharesOutstanding,"Entity Common Stock, Shares Outstanding",shares,2010-07-09,NaN,913562880.0,0001193125-10-162840,10-Q,2010-07-21,CY2010Q2I,AAPL
6,0000320193,Apple Inc.,dei,EntityCommonStockSharesOutstanding,"Entity Common Stock, Shares Outstanding",shares,2010-10-15,NaN,917307099.0,0001193125-10-238044,10-K,2010-10-27,CY2010Q3I,AAPL
7,0000320193,Apple Inc.,dei,EntityCommonStockSharesOutstanding,"Entity Common Stock, Shares Outstanding",shares,2011-01-07,NaN,921278012.0,0001193125-11-010144,10-Q,2011-01-19,CY2010Q4I,AAPL
8,0000320193,Apple Inc.,dei,EntityCommonStockSharesOutstanding,"Entity Common Stock, Shares Outstanding",shares,2011-04-08,NaN,924754561.0,0001193125-11-104388,10-Q,2011-04-21,CY2011Q1I,AAPL
9,0000320193,Apple Inc.,dei,EntityCommonStockSharesOutstanding,"Entity Common Stock, Shares Outstanding",shares,2011-07-08,NaN,927090886.0,0001193125-11-192493,10-Q,2011-07-20,CY2011Q2I,AAPL


---
## 4. Spot Checks — What's in the Raw Data?

In [11]:
# Namespace breakdown — how many rows per namespace?
print("Rows by namespace:")
print(facts_df["namespace"].value_counts().to_string())
print()

# Concept count per namespace
for ns, grp in facts_df.groupby("namespace"):
    print(f"  {ns:<12}  {grp['concept'].nunique():>5} unique concepts  |  {len(grp):>8,} rows")

Rows by namespace:
namespace
us-gaap    258495
dei           716
invest         70
ffd            49
ecd            35
srt            21

  dei               3 unique concepts  |       716 rows
  ecd               7 unique concepts  |        35 rows
  ffd               1 unique concepts  |        49 rows
  invest            1 unique concepts  |        70 rows
  srt               2 unique concepts  |        21 rows
  us-gaap        2393 unique concepts  |   258,495 rows


In [12]:
# Annual revenue — filter in the DataFrame using concept name + namespace
# (no hardcoded list needed — query by name just like you would in DuckDB)
REVENUE_CONCEPTS = ["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax"]

revenue = (
    facts_df[
        (facts_df["namespace"] == "us-gaap") &
        (facts_df["concept"].isin(REVENUE_CONCEPTS)) &
        (facts_df["unit"] == "USD") &
        (facts_df["form"].isin(["10-K", "10-K/A"])) &
        (facts_df["start"].notna())   # annual = has a start date
    ]
    .assign(days=lambda x: (x["end"] - pd.to_datetime(x["start"])).dt.days)
    .query("355 <= days <= 375")      # annual period
    .sort_values("end", ascending=False)
    .groupby("ticker").head(5)
    [["ticker", "end", "val", "form", "filed", "frame"]]
    .assign(revenue_bn=lambda x: (x["val"] / 1e9).round(2))
)
revenue

,ticker,end,val,form,filed,frame,revenue_bn
257967,BRK-B,2025-12-31,3.714440e+11,10-K,2026-03-02,CY2025,371.44
81447,AMZN,2025-12-31,7.169240e+11,10-K,2026-02-06,CY2025,716.92
141574,TSLA,2025-12-31,9.482700e+10,10-K,2026-01-29,CY2025,94.83
119278,META,2025-12-31,2.009660e+11,10-K,2026-01-29,CY2025,200.97
141850,TSLA,2025-12-31,9.482700e+10,10-K,2026-01-29,CY2025,94.83
186140,JPM,2025-12-31,1.824470e+11,10-K,2026-02-13,CY2025,182.45
102614,GOOG,2025-12-31,4.028360e+11,10-K,2026-02-05,CY2025,402.84
257579,BRK-B,2025-12-31,2.472440e+11,10-K,2026-03-02,CY2025,247.24
238883,XOM,2025-12-31,3.322380e+11,10-K,2026-02-18,CY2025,332.24
217008,JNJ,2025-12-28,9.419300e+10,10-K,2026-02-11,CY2025,94.19


In [13]:
# Revenue pivot by fiscal year
pivot = (
    revenue
    .assign(year=revenue["end"].dt.year)
    .drop_duplicates(["ticker", "year"])
    .pivot_table(index="ticker", columns="year", values="revenue_bn", aggfunc="first")
    .sort_index()
)
print("Annual Revenue ($B)")
pivot

Annual Revenue ($B)


year,2023,2024,2025
ticker,,,
AAPL,383.28,391.04,416.16
AMZN,574.78,637.96,716.92
BRK-B,NaN,249.71,371.44
GOOG,307.39,350.02,402.84
JNJ,85.16,88.82,94.19
JPM,158.10,177.56,182.45
META,134.90,164.50,200.97
MSFT,211.92,245.12,281.72
TSLA,NaN,97.69,94.83


In [14]:
# DEI namespace — shares outstanding and entity info (previously dropped entirely)
dei_df = facts_df[facts_df["namespace"] == "dei"].copy()
print(f"DEI rows: {len(dei_df):,}")
print("\nDEI concepts available:")
print(dei_df["concept"].value_counts().head(20).to_string())

DEI rows: 716

DEI concepts available:
concept
EntityCommonStockSharesOutstanding    478
EntityPublicFloat                     190
EntityListingParValuePerShare          48


---
## 5. Data Quality Summary

In [15]:
# Submissions metadata field coverage
print("Submissions metadata field coverage:")
for col in submissions_df.columns:
    filled = submissions_df[col].notna().sum()
    print(f"  {col:<45} {100*filled/len(submissions_df):5.1f}%")

Submissions metadata field coverage:
  cik                                           100.0%
  entityType                                    100.0%
  sic                                           100.0%
  sicDescription                                100.0%
  ownerOrg                                      100.0%
  insiderTransactionForOwnerExists              100.0%
  insiderTransactionForIssuerExists             100.0%
  name                                          100.0%
  tickers                                       100.0%
  exchanges                                     100.0%
  ein                                           100.0%
  lei                                             0.0%
  description                                   100.0%
  website                                       100.0%
  investorWebsite                               100.0%
  category                                      100.0%
  fiscalYearEnd                                 100.0%
  stateOfIncorporation      

In [16]:
# Overall summary
print("=" * 60)
print("PHASE 1 — RAW DATA SUMMARY (all fields, no filtering)")
print("=" * 60)
print(f"  Tickers snapshot   : {len(tickers_df):>8,} rows")
print(f"  NYSE + Nasdaq      : {len(listed):>8,} tickers")
print()
print(f"  Submissions        : {len(submissions_df):>8} companies  x {submissions_df.shape[1]} columns")
print(f"  Filings index      : {len(filings_df):>8,} filing rows x {filings_df.shape[1]} columns")
print()
print(f"  Facts total        : {len(facts_df):>8,} rows")
print(f"  Unique concepts    : {facts_df['concept'].nunique():>8,}")
print(f"  Namespaces         : {sorted(facts_df['namespace'].unique())}")
print(f"  With frame tag     : {facts_df['frame'].notna().sum():>8,} rows")
print(f"  USD facts          : {(facts_df['unit']=='USD').sum():>8,} rows")
print()
print("SEC compliance:")
print(f"  User-Agent         : {USER_AGENT}")
print(f"  Rate limit         : 8 req/s  (SEC limit = 10 req/s per IP)")

PHASE 1 — RAW DATA SUMMARY (all fields, no filtering)
  Tickers snapshot   :   10,433 rows
  NYSE + Nasdaq      :    7,521 tickers

  Submissions        :       10 companies  x 42 columns
  Filings index      :   32,678 filing rows x 18 columns

  Facts total        :  259,386 rows
  Unique concepts    :    2,404
  Namespaces         : ['dei', 'ecd', 'ffd', 'invest', 'srt', 'us-gaap']
  With frame tag     :  106,966 rows
  USD facts          :  233,456 rows

SEC compliance:
  User-Agent         : n/a prototyping paul.ananth@yahoo.com
  Rate limit         : 8 req/s  (SEC limit = 10 req/s per IP)
